# DS Coding — Pandas Drill #2 (2026-06-24)

MLE-level data manipulation: **merge/join + groupby + a derived metric**. Realistic, not a puzzle.

## Scenario
You have two tables from a streaming app:
- `users` — one row per user (with their signup plan)
- `events` — watch events (a user can have many; some users have none)

## Task
Produce a per-**plan** summary table with:
- `n_users` — number of users on that plan
- `total_watch_secs` — total watch seconds by users on that plan (0 if none)
- `avg_watch_per_user` — total_watch_secs / n_users

**Watch out:** some users have NO events. They must still be counted in `n_users` and contribute 0 watch time (this is the part that tests whether you pick the right join).

Run setup, write your solution in the answer cell, check against *Expected* at the bottom.

In [42]:
import pandas as pd

users = pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5],
    'plan':    ['premium', 'free', 'premium', 'free', 'free'],
})

events = pd.DataFrame({
    'user_id':    [1, 1, 2, 3, 3, 3],
    'watch_secs': [100, 200, 50, 400, 100, 100],
})
# Note: users 4 and 5 (both 'free') have NO events.
users, events

(   user_id     plan
 0        1  premium
 1        2     free
 2        3  premium
 3        4     free
 4        5     free,
    user_id  watch_secs
 0        1         100
 1        1         200
 2        2          50
 3        3         400
 4        3         100
 5        3         100)

## Your solution
Think about: which **join** keeps users with no events? How do you avoid NaN turning into wrong sums/counts?

In [49]:
# Your answer here
result = None
total_watch = events.groupby(['user_id'])['watch_secs'].sum().to_dict()
users['total_watch'] = users['user_id'].map(total_watch)
users['total_watch'] = users['total_watch'].fillna(0)
result = users.groupby(['plan']).agg({'plan' : ['count'] , 'total_watch' :['sum', 'mean']})


result

plan total_watch            
        count         sum        mean
plan                                 
free        3        50.0   16.666667
premium     2       900.0  450.000000

## Expected output (verify your logic)
```
      plan  n_users  total_watch_secs  avg_watch_per_user
      free        3               50            16.666667
   premium        2              900           450.000000
```
Check the logic:
- **free**: users 2,4,5 = 3 users. Watch: user2=50, user4=0, user5=0 -> 50 total -> 50/3 = 16.67
- **premium**: users 1,3 = 2 users. Watch: user1=300, user3=600 -> 900 total -> 900/2 = 450